# Music Annotation Project V8.8B
## Notebook 53: Operational M5_REMOVE Builder

**Runtime:** CPU. Select **Runtime → Run all**.

Notebook 53 consumes the frozen Notebook 52 policy identity and builds operational M5_REMOVE page artefacts from an observed page, an authoritative intervention mask/M3 mask, and SAFE_REMOVE decisions produced by the frozen Notebook 50 policy.

### Fixed contract
- `M5_REMOVE = M3 & ~SAFE_REMOVE`.
- SAFE_PROTECT is not enabled.
- No retraining, recalibration or threshold movement.
- Clean truth is prohibited from inference.
- SAFE_REMOVE must be a subset of M3.
- Unsupported, missing, invalid or uncertain coordinates retain exact M3 treatment.
- Pixels outside the authoritative mask remain byte-for-byte equal to the observed page.
- This notebook does not replace Notebook 09 by itself. Notebook 54 must prove exact V8.7 equivalence before shadow use.

### Two input routes
1. **Frozen regression route:** uses Notebook 50's persisted OOF pixel decisions for pages in the registered V8.7 roster. This is the mandatory first execution route.
2. **Operational feature route:** accepts a page-level predictor table produced by the exact frozen Notebook 50 observable-feature builder. This route stops unless a compatible deployable REMOVE scorer can be resolved and replay-verified.

Binary model, calibration, parquet and NPZ artefacts remain in Google Drive and are validated in place.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pandas numpy pyarrow joblib scikit-learn opencv-python-headless


## 1. Configuration and output identity


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, zipfile, sys, platform, os, re, traceback
import numpy as np
import pandas as pd
import cv2, joblib

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
N52_ROOT=ROOT/'m5_remove_policy_freeze_v8_8a'
OUTPUT_ROOT=ROOT/'m5_remove_operational_builder_v8_8b'

# Optional exact run pins. Leave None to use latest complete runs.
NOTEBOOK52_RUN_OVERRIDE=None
NOTEBOOK50_RUN_OVERRIDE=None

# Mandatory first execution is registered V8.7 replay.
INPUT_MODE='REGISTERED_REPLAY'  # REGISTERED_REPLAY or OPERATIONAL_FEATURE_TABLES

# Operational inputs, used only when INPUT_MODE == OPERATIONAL_FEATURE_TABLES.
OPERATIONAL_ROOT=ROOT/'v8_8_operational_inputs'
OBSERVED_DIR=OPERATIONAL_ROOT/'observed'
M3_MASK_DIR=OPERATIONAL_ROOT/'m3_masks'
FEATURE_DIR=OPERATIONAL_ROOT/'features'
SOURCE_MANIFEST=OPERATIONAL_ROOT/'page_manifest.csv'

# Reconstruction/output contract. Telea is the initial frozen operational method;
# Notebook 54 must compare it against the registered Notebook 51 arm.
RECONSTRUCTION_METHOD='telea'
INPAINT_RADIUS=3
CROP_MARGIN=48

SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook53_v8_8b'
OUT=OUTPUT_ROOT/RUN
INPUTS=OUT/'inputs'; TABLES=OUT/'tables'; MASKS=OUT/'masks'; PAGES=OUT/'pages'; AUDIT=OUT/'audit'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,TABLES,MASKS,PAGES,AUDIT,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
print('Mode:',INPUT_MODE); print('Output:',OUT)


## 2. Helpers and fail-closed source resolution


In [ ]:
def require(condition,message):
    if not bool(condition): raise RuntimeError(message)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()

def read_json(path): return json.loads(Path(path).read_text())

def atomic_json(path,value):
    tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)

def image_map(folder):
    return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}

def latest_complete(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def resolve_unique(raw,roots=()):
    p=Path(str(raw)); candidates=[p]+[Path(r)/p for r in roots]+[Path(r)/p.name for r in roots]
    unique={}
    for x in candidates:
        if x.exists(): unique[str(x.resolve())]=x.resolve()
    hits=list(unique.values())
    require(len(hits)==1,f'Expected one file for {raw}; found {len(hits)} unique files: {hits}')
    return hits[0]

def binary(a): return np.asarray(a)>0


## 3. Resolve and verify the passed Notebook 52 policy


In [ ]:
N52_REQUIRED=['notebook52_verdict.json','policy_bundle/policy_manifest.json','tables/m5_remove_golden_master_semantic_hashes.csv','tables/notebook50_full_artifact_inventory.csv']
N52=Path(NOTEBOOK52_RUN_OVERRIDE) if NOTEBOOK52_RUN_OVERRIDE else latest_complete(N52_ROOT,N52_REQUIRED)
require(N52 is not None and N52.exists(),'No complete Notebook 52 V8.8A run found.')
verdict52=read_json(N52/'notebook52_verdict.json'); policy=read_json(N52/'policy_bundle'/'policy_manifest.json')
require(verdict52.get('verdict')=='PASS_M5_REMOVE_FREEZE_READY_FOR_OPERATIONAL_BUILDER',f'Notebook 52 did not pass: {verdict52}')
require(policy.get('policy')=='M5_REMOVE','Notebook 52 policy is not M5_REMOVE.')
require(policy.get('safe_protect_runtime_enabled') is False,'SAFE_PROTECT must remain disabled.')
require(policy.get('retraining_performed') is False and policy.get('recalibration_performed') is False and policy.get('threshold_changed') is False,'Notebook 52 policy mutation detected.')
EXPECTED_POLICY_ID=policy['policy_id']
N50=Path(NOTEBOOK50_RUN_OVERRIDE) if NOTEBOOK50_RUN_OVERRIDE else Path(policy['authoritative_notebook50_run'])
require(N50.exists(),f'Authoritative Notebook 50 run missing: {N50}')
print('Notebook 52:',N52); print('Notebook 50:',N50); print('Policy ID:',EXPECTED_POLICY_ID)


## 4. Resolve Notebook 50 schema, predictions and decision encoding


In [ ]:
schema=read_json(N50/'inputs'/'predictor_schema.json')
PREDICTORS=schema.get('ordered_predictors',schema.get('predictors',schema.get('ordered_features',[])))
require(len(PREDICTORS)>0,'Frozen ordered predictor schema is empty.')

pred_path=N50/'tables'/'all_m3_oof_predictions_and_decisions.parquet'
require(pred_path.exists(),f'Missing Notebook 50 decision table: {pred_path}')
pred=pd.read_parquet(pred_path)
required={'page','y','x','safe_remove'}
require(required.issubset(pred.columns),f'Notebook 50 prediction table missing: {sorted(required-set(pred.columns))}')
pred['page']=pred.page.astype(str); pred['y']=pd.to_numeric(pred.y,errors='raise').astype(int); pred['x']=pd.to_numeric(pred.x,errors='raise').astype(int)
pred['safe_remove']=pred.safe_remove.astype(bool)
require(not pred.duplicated(['page','y','x']).any(),'Duplicate Notebook 50 page/y/x decision rows.')

map_inventory=pd.read_csv(N50/'tables'/'m5_map_inventory.csv'); map_inventory['page']=map_inventory.page.astype(str)
require({'page','file'}.issubset(map_inventory.columns),'Notebook 50 m5_map_inventory lacks page/file.')

# Infer and verify m5_decision_map encoding from pages
# that actually contain SAFE_REMOVE assertions.
#
# Pages with zero SAFE_REMOVE rows are valid M3-fallback pages
# and must not be used to infer the SAFE_REMOVE integer code.

encoding_rows = []

for page, g in pred.groupby('page'):

    inv = map_inventory[
        map_inventory.page == page
    ]

    require(
        len(inv) == 1,
        f'{page}: expected one map inventory row, '
        f'found {len(inv)}'
    )

    p = resolve_unique(
        inv.iloc[0]['file'],
        [N50, N50/'maps', N50/'masks']
    )

    with np.load(p, allow_pickle=False) as z:

        require(
            {'m5_decision_map', 'reason_map'}.issubset(z.files),
            f'{p}: missing decision arrays; keys={z.files}'
        )

        dm = np.asarray(z['m5_decision_map'])
        rm = np.asarray(z['reason_map'])

    require(
        dm.shape == rm.shape and dm.ndim == 2,
        f'{page}: decision/reason shape failure.'
    )

    require(
        (g.y >= 0).all()
        and (g.x >= 0).all()
        and (g.y < dm.shape[0]).all()
        and (g.x < dm.shape[1]).all(),
        f'{page}: prediction coordinate outside saved '
        f'decision canvas.'
    )

    values = dm[
        g.y.to_numpy(),
        g.x.to_numpy()
    ]

    sr_mask = g.safe_remove.to_numpy(dtype=bool)

    safe_remove_rows = int(sr_mask.sum())

    # -----------------------------------------------
    # Page contains one or more SAFE_REMOVE decisions
    # -----------------------------------------------

    if safe_remove_rows > 0:

        srvals = values[sr_mask]
        unique_sr = np.unique(srvals)

        require(
            len(unique_sr) == 1,
            f'{page}: SAFE_REMOVE rows map to multiple '
            f'decision codes {unique_sr}'
        )

        safe_remove_code = int(unique_sr[0])

    # -----------------------------------------------
    # Zero SAFE_REMOVE is a legitimate page outcome.
    # There is no code to infer from this page.
    # -----------------------------------------------

    else:

        safe_remove_code = None

    encoding_rows.append({
        'page': page,

        'safe_remove_rows':
            safe_remove_rows,

        'safe_remove_code':
            safe_remove_code,

        'all_decision_values':
            '|'.join(
                map(str, np.unique(dm).tolist())
            ),

        'decision_shape':
            str(dm.shape),

        'zero_safe_remove_page':
            bool(safe_remove_rows == 0),
    })


encoding = pd.DataFrame(encoding_rows)

# ---------------------------------------------------
# Infer the global SAFE_REMOVE code only from pages
# where SAFE_REMOVE was actually asserted.
# ---------------------------------------------------

observed_codes = (
    encoding.loc[
        encoding.safe_remove_rows > 0,
        'safe_remove_code'
    ]
    .dropna()
    .astype(int)
    .unique()
)

require(
    len(observed_codes) == 1,
    'Could not identify one consistent Notebook 50 '
    f'SAFE_REMOVE code. Observed codes: {observed_codes}'
)

SAFE_REMOVE_CODE = int(observed_codes[0])

# Every page which DID contain SAFE_REMOVE must agree
# with that global code.
require(
    (
        encoding.loc[
            encoding.safe_remove_rows > 0,
            'safe_remove_code'
        ].astype(int)
        == SAFE_REMOVE_CODE
    ).all(),
    'Notebook 50 SAFE_REMOVE encoding is inconsistent '
    'between pages.'
)

encoding.to_csv(
    AUDIT/'decision_encoding_audit.csv',
    index=False
)

print('Predictors:', len(PREDICTORS))
print('SAFE_REMOVE decision code:', SAFE_REMOVE_CODE)
print(
    'Pages with SAFE_REMOVE:',
    int((encoding.safe_remove_rows > 0).sum())
)
print(
    'Pages with zero SAFE_REMOVE:',
    int((encoding.safe_remove_rows == 0).sum())
)

display(
    encoding[
        [
            'page',
            'safe_remove_rows',
            'safe_remove_code',
            'all_decision_values',
            'zero_safe_remove_page',
        ]
    ]
)

## 5. Resolve page inputs for the selected route


In [ ]:
def find_page_image(page,folders):
    hits={}
    for folder in folders:
        if not Path(folder).exists(): continue
        for p in Path(folder).glob(page+'.*'):
            if p.is_file() and p.suffix.lower() in SUPPORTED: hits[str(p.resolve())]=p.resolve()
    found=list(hits.values())
    require(len(found)==1,f'{page}: expected one image across {folders}; found {found}')
    return found[0]

def discover_reference_page_inputs(page):
    observed=find_page_image(page,[ROOT/'aligned',ROOT/'source'/'annotated_only',ROOT/'source'/'annotated'])
    # M3 may be persisted as a page image or semantic array in the Notebook 51/50 lineage.
    m3_candidates=[]
    for base in [N50,ROOT/'residual_mask_crossover_v8_7',ROOT/'reconstruction_geometry_benchmark_v7_5']:
        if not Path(base).exists(): continue
        for p in Path(base).rglob('*'):
            if p.is_file() and page.lower() in p.stem.lower() and ('m3' in p.stem.lower() or 'mask' in p.stem.lower()) and p.suffix.lower() in SUPPORTED:
                m3_candidates.append(p.resolve())
    m3_candidates=list(dict.fromkeys(m3_candidates))
    require(len(m3_candidates)==1,f'{page}: expected one reference M3 mask; found {len(m3_candidates)}. Set operational inputs or remove ambiguous copies. Candidates={m3_candidates[:20]}')
    return observed,m3_candidates[0],None

if INPUT_MODE=='REGISTERED_REPLAY':
    roster=sorted(pred.page.unique())
    source_rows=[]
    for page in roster:
        obs,m3,_=discover_reference_page_inputs(page)
        source_rows.append({'page':page,'observed_path':str(obs),'m3_path':str(m3),'feature_path':str(pred_path),'source_group':str(pred.loc[pred.page==page,'source_group'].iloc[0]) if 'source_group' in pred.columns else ''})
    sources=pd.DataFrame(source_rows)
elif INPUT_MODE=='OPERATIONAL_FEATURE_TABLES':
    require(SOURCE_MANIFEST.exists(),f'Missing operational page manifest: {SOURCE_MANIFEST}')
    sources=pd.read_csv(SOURCE_MANIFEST,dtype=str)
    require({'page','source_group'}.issubset(sources.columns),'Operational manifest needs page/source_group.')
    sources['page']=sources.page.astype(str)
    rows=[]
    for _,r in sources.iterrows():
        page=r.page; obs=find_page_image(page,[OBSERVED_DIR]); m3=find_page_image(page,[M3_MASK_DIR]); fp=FEATURE_DIR/f'{page}.parquet'
        require(fp.exists(),f'{page}: missing operational feature table {fp}')
        rows.append({'page':page,'source_group':r.source_group,'observed_path':str(obs),'m3_path':str(m3),'feature_path':str(fp)})
    sources=pd.DataFrame(rows)
else: raise ValueError(INPUT_MODE)
require(not sources.page.duplicated().any(),'Input pages must be unique.')
sources.to_csv(INPUTS/'resolved_page_inputs.csv',index=False)
print('Pages:',len(sources))


## 6. Freeze-compatible REMOVE scoring

The registered replay route consumes Notebook 50's exact persisted OOF decisions. The operational feature route requires a separately serialised deployable scorer whose schema and policy identity match Notebook 52. Fold OOF decisions are not silently converted into an unseen-page ensemble.


In [ ]:
def find_deployable_scorer():
    candidates=[]
    for p in N50.rglob('*.joblib'):
        name=p.name.lower()
        if ('deploy' in name or 'final' in name or 'full' in name) and ('remove' in name or 'm5' in name): candidates.append(p.resolve())
    candidates=list(dict.fromkeys(candidates))
    require(len(candidates)==1,'Operational feature mode requires exactly one explicitly deployable full-development REMOVE scorer. '
            f'Found {candidates}. Do not substitute OOF fold models or retrain in Notebook 53.')
    return candidates[0]

def decisions_for_page(page,feature_path):
    if INPUT_MODE=='REGISTERED_REPLAY':
        q=pred[pred.page==page].copy()
        return q[['page','y','x','safe_remove']+[c for c in ['p_remove','remove_probability','m5_reason'] if c in q.columns]]
    features=pd.read_parquet(feature_path)
    require({'page','y','x'}.issubset(features.columns),f'{page}: feature table lacks page/y/x.')
    require(not features.duplicated(['page','y','x']).any(),f'{page}: duplicate feature coordinates.')
    missing=[c for c in PREDICTORS if c not in features.columns]
    require(not missing,f'{page}: missing frozen predictors: {missing}')
    X=features.loc[:,PREDICTORS].apply(pd.to_numeric,errors='coerce')
    require(not np.isinf(X.to_numpy(float)).any(),f'{page}: infinite predictor value.')
    scorer_path=find_deployable_scorer(); scorer=joblib.load(scorer_path)
    require(hasattr(scorer,'predict_proba') or hasattr(scorer,'predict'),f'{scorer_path}: scorer has no prediction method.')
    if hasattr(scorer,'predict_proba'):
        prob=np.asarray(scorer.predict_proba(X))[:,1]
        # Threshold must be discoverable from Notebook 50 verdict/control records.
        threshold=None
        for obj in [read_json(N50/'notebook50_verdict.json'),read_json(N50/'selected_models.json')]:
            for key in ['remove_threshold','safe_remove_threshold','threshold_remove']:
                if key in obj: threshold=float(obj[key])
        require(threshold is not None,'Frozen REMOVE threshold could not be resolved from Notebook 50 records.')
        selected=prob>=threshold
    else:
        selected=np.asarray(scorer.predict(X)).astype(bool); prob=np.full(len(selected),np.nan)
    return pd.DataFrame({'page':page,'y':features.y.astype(int),'x':features.x.astype(int),'safe_remove':selected,'p_remove':prob})


## 7. Canonical M5_REMOVE masks and correction outputs


In [ ]:
def padded_components(mask,margin=CROP_MARGIN,min_area=8):
    n,lab,stats,_=cv2.connectedComponentsWithStats(mask.astype(np.uint8),8); h,w=mask.shape; boxes=[]
    for i in range(1,n):
        x,y,bw,bh,a=stats[i]
        if a>=min_area: boxes.append((max(0,x-margin),max(0,y-margin),min(w,x+bw+margin),min(h,y+bh+margin)))
    changed=True
    while changed:
        changed=False; merged=[]
        while boxes:
            ax0,ay0,ax1,ay1=boxes.pop(); rest=[]
            for bx0,by0,bx1,by1 in boxes:
                if not (ax1<bx0 or bx1<ax0 or ay1<by0 or by1<ay0):
                    ax0=min(ax0,bx0); ay0=min(ay0,by0); ax1=max(ax1,bx1); ay1=max(ay1,by1); changed=True
                else: rest.append((bx0,by0,bx1,by1))
            boxes=rest; merged.append((ax0,ay0,ax1,ay1))
        boxes=merged
    return boxes

def background_surface(observed,mask,sigma=45.0):
    visible=(~mask)&(observed>=200); values=observed.astype(np.float32)*visible; weights=visible.astype(np.float32)
    num=cv2.GaussianBlur(values,(0,0),sigma); den=cv2.GaussianBlur(weights,(0,0),sigma); fallback=cv2.GaussianBlur(observed,(0,0),sigma)
    return np.clip(np.where(den>0.02,num/np.maximum(den,1e-4),fallback),0,255).astype(np.uint8)

def reconstruct_residual(observed,residual):
    result=observed.copy(); mask=(residual.astype(np.uint8)*255)
    for x0,y0,x1,y1 in padded_components(residual):
        im=observed[y0:y1,x0:x1]; mk=mask[y0:y1,x0:x1]
        if RECONSTRUCTION_METHOD=='telea': cand=cv2.inpaint(im,mk,INPAINT_RADIUS,cv2.INPAINT_TELEA)
        elif RECONSTRUCTION_METHOD=='navier_stokes': cand=cv2.inpaint(im,mk,INPAINT_RADIUS,cv2.INPAINT_NS)
        else: raise ValueError(RECONSTRUCTION_METHOD)
        local=result[y0:y1,x0:x1]; local[mk>0]=cand[mk>0]; result[y0:y1,x0:x1]=local
    require(np.array_equal(result[~residual],observed[~residual]),'Residual reconstruction changed non-residual coordinates.')
    return result

def build_output(observed,m3,safe_remove):
    require(observed.shape==m3.shape==safe_remove.shape,f'Shape mismatch observed={observed.shape}, M3={m3.shape}, SAFE_REMOVE={safe_remove.shape}')
    require(not np.any(safe_remove & ~m3),'SAFE_REMOVE is not a subset of M3.')
    m5=m3 & ~safe_remove
    reconstructed=reconstruct_residual(observed,m5)
    paper=background_surface(observed,m3)
    out=reconstructed.copy(); out[safe_remove]=paper[safe_remove]
    require(np.array_equal(out[~m3],observed[~m3]),'Output changed outside authoritative M3 mask.')
    return m5,out,paper,reconstructed


## 8. Execute every page and persist separate audit artefacts


In [ ]:
page_rows=[]; decision_rows=[]
for _,src in sources.iterrows():
    page=str(src.page); observed=cv2.imread(str(src.observed_path),0); m3_image=cv2.imread(str(src.m3_path),0)
    require(observed is not None and m3_image is not None,f'{page}: unreadable observed/M3 image.')
    m3=m3_image>0; require(observed.shape==m3.shape,f'{page}: observed/M3 shape mismatch.')
    decisions=decisions_for_page(page,Path(src.feature_path)); decisions['page']=decisions.page.astype(str)
    require((decisions.y>=0).all() and (decisions.x>=0).all() and (decisions.y<observed.shape[0]).all() and (decisions.x<observed.shape[1]).all(),f'{page}: decision coordinate outside page.')
    safe_remove=np.zeros(observed.shape,bool); q=decisions[decisions.safe_remove.astype(bool)]; safe_remove[q.y.to_numpy(int),q.x.to_numpy(int)]=True
    require(not np.any(safe_remove & ~m3),f'{page}: selected SAFE_REMOVE outside M3.')
    m5,out,paper,reconstructed=build_output(observed,m3,safe_remove)
    d=PAGES/page; d.mkdir(parents=True,exist_ok=False)
    cv2.imwrite(str(d/'observed.png'),observed); cv2.imwrite(str(d/'m3.png'),m3.astype(np.uint8)*255); cv2.imwrite(str(d/'safe_remove.png'),safe_remove.astype(np.uint8)*255); cv2.imwrite(str(d/'m5_remove.png'),m5.astype(np.uint8)*255); cv2.imwrite(str(d/'residual_reconstruction.png'),reconstructed); cv2.imwrite(str(d/'paper_estimate.png'),paper); cv2.imwrite(str(d/'corrected.png'),out)
    diff=np.zeros_like(observed); diff[out!=observed]=255; cv2.imwrite(str(d/'difference_from_observed.png'),diff)
    np.savez_compressed(MASKS/f'{page}_m5_remove_operational.npz',M3=m3.astype(np.uint8),SAFE_REMOVE=safe_remove.astype(np.uint8),M5_REMOVE=m5.astype(np.uint8))
    decisions.to_parquet(TABLES/f'{page}_safe_remove_decisions.parquet',index=False)
    for _,r in decisions.iterrows(): decision_rows.append({'page':page,'y':int(r.y),'x':int(r.x),'safe_remove':bool(r.safe_remove),'p_remove':r.get('p_remove',np.nan)})
    page_rows.append({'page':page,'source_group':src.source_group,'observed_path':src.observed_path,'m3_path':src.m3_path,'feature_path':src.feature_path,'shape':str(observed.shape),'M3_pixels':int(m3.sum()),'SAFE_REMOVE_pixels':int(safe_remove.sum()),'M5_REMOVE_pixels':int(m5.sum()),'outside_M3_drift':int(np.sum(out[~m3]!=observed[~m3])),'M3_sha256':array_hash(m3.astype(np.uint8)),'SAFE_REMOVE_sha256':array_hash(safe_remove.astype(np.uint8)),'M5_REMOVE_sha256':array_hash(m5.astype(np.uint8)),'corrected_sha256':array_hash(out),'policy_id':EXPECTED_POLICY_ID})
    print('Built',page,'M3',int(m3.sum()),'SAFE_REMOVE',int(safe_remove.sum()),'M5_REMOVE',int(m5.sum()))
page_df=pd.DataFrame(page_rows); decisions_df=pd.DataFrame(decision_rows)
page_df.to_csv(TABLES/'operational_page_inventory.csv',index=False); decisions_df.to_parquet(TABLES/'all_operational_safe_remove_decisions.parquet',index=False)
require(len(page_df)==len(sources),'Not every input page completed.')
require((page_df.outside_M3_drift==0).all(),'Outside-M3 immutability failure.')


## 9. Registered-replay equivalence against Notebook 50 semantic maps


In [ ]:
replay_rows=[]
if INPUT_MODE=='REGISTERED_REPLAY':
    for _,r in page_df.iterrows():
        page=r.page; inv=map_inventory[map_inventory.page==page]; p=resolve_unique(inv.iloc[0]['file'],[N50,N50/'maps',N50/'masks'])
        with np.load(p,allow_pickle=False) as z: dm=np.asarray(z['m5_decision_map'])
        saved=(dm==SAFE_REMOVE_CODE)
        built=np.load(MASKS/f'{page}_m5_remove_operational.npz')['SAFE_REMOVE']>0
        # Notebook 50 may have a bottom/right truncated decision canvas. Pad only at origin.
        require(saved.shape[0]<=built.shape[0] and saved.shape[1]<=built.shape[1],f'{page}: saved decision canvas exceeds page.')
        padded=np.zeros(built.shape,bool); padded[:saved.shape[0],:saved.shape[1]]=saved
        xor=int(np.logical_xor(padded,built).sum())
        replay_rows.append({'page':page,'saved_shape':str(saved.shape),'operational_shape':str(built.shape),'safe_remove_xor_pixels':xor,'exact':xor==0,'saved_semantic_sha256':array_hash(saved.astype(np.uint8)),'padded_semantic_sha256':array_hash(padded.astype(np.uint8)),'operational_semantic_sha256':array_hash(built.astype(np.uint8))})
    replay_df=pd.DataFrame(replay_rows); replay_df.to_csv(TABLES/'notebook50_safe_remove_replay.csv',index=False)
    require(replay_df.exact.all(),f'Notebook 50 SAFE_REMOVE replay failure on: {replay_df.loc[~replay_df.exact,"page"].tolist()}')
    print('Exact SAFE_REMOVE replay pages:',len(replay_df))
else:
    replay_df=pd.DataFrame(columns=['page','exact']); replay_df.to_csv(TABLES/'notebook50_safe_remove_replay.csv',index=False)


## 10. Fail-closed V8.8B verdict


In [ ]:
gates={
 'notebook52_policy_pass':verdict52.get('verdict')=='PASS_M5_REMOVE_FREEZE_READY_FOR_OPERATIONAL_BUILDER',
 'policy_identity_exact':policy.get('policy_id')==EXPECTED_POLICY_ID,
 'remove_only':policy.get('safe_protect_runtime_enabled') is False,
 'all_pages_completed':len(page_df)==len(sources),
 'safe_remove_subset_m3':bool((page_df.SAFE_REMOVE_pixels<=page_df.M3_pixels).all()),
 'outside_m3_immutable':bool((page_df.outside_M3_drift==0).all()),
 'notebook50_replay_exact':bool(replay_df.exact.all()) if INPUT_MODE=='REGISTERED_REPLAY' and len(replay_df) else None,
}
required_gate_names=['notebook52_policy_pass','policy_identity_exact','remove_only','all_pages_completed','safe_remove_subset_m3','outside_m3_immutable']
if INPUT_MODE=='REGISTERED_REPLAY': required_gate_names.append('notebook50_replay_exact')
first_failed=next((k for k in required_gate_names if gates.get(k) is not True),None)
verdict='PASS_REGISTERED_REPLAY_READY_FOR_NOTEBOOK54_EQUIVALENCE' if first_failed is None and INPUT_MODE=='REGISTERED_REPLAY' else ('PASS_OPERATIONAL_BUILD_READY_FOR_NOTEBOOK54_EQUIVALENCE' if first_failed is None else 'FAIL_V8_8B_OPERATIONAL_BUILDER')
record={'created_utc':datetime.now(timezone.utc).isoformat(),'verdict':verdict,'first_failed_gate':first_failed,'gates':gates,'mode':INPUT_MODE,'policy_id':EXPECTED_POLICY_ID,'notebook52_run':str(N52),'notebook50_run':str(N50),'pages':len(page_df),'reconstruction_method':RECONSTRUCTION_METHOD,'promotion_authority':False,'meaning':'PASS permits Notebook 54 exact-equivalence testing only. It does not replace Notebook 09 or authorise automatic output.'}
atomic_json(OUT/'notebook53_verdict.json',record); pd.DataFrame([{'gate':k,'passed':v} for k,v in gates.items()]).to_csv(TABLES/'notebook53_gate_results.csv',index=False)
print(json.dumps(record,indent=2))
if first_failed: raise RuntimeError(f'Notebook 53 failed closed at {first_failed}.')


## 11. Identity, source fingerprint and upload package


In [ ]:
environment={'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}
identity={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.8B-notebook53-operational-m5-remove-builder','mode':INPUT_MODE,'policy_id':EXPECTED_POLICY_ID,'notebook52_verdict_sha256':sha256_file(N52/'notebook52_verdict.json'),'policy_manifest_sha256':sha256_file(N52/'policy_bundle'/'policy_manifest.json'),'notebook50_predictions_sha256':sha256_file(pred_path),'predictor_schema_sha256':sha256_file(N50/'inputs'/'predictor_schema.json'),'safe_remove_code':SAFE_REMOVE_CODE,'reconstruction_method':RECONSTRUCTION_METHOD,'inpaint_radius':INPAINT_RADIUS,'crop_margin':CROP_MARGIN,'pages':len(page_df),'page_inventory_sha256':sha256_file(TABLES/'operational_page_inventory.csv'),'decision_table_sha256':sha256_file(TABLES/'all_operational_safe_remove_decisions.parquet'),'retrained':False,'recalibrated':False,'threshold_changed':False,'safe_protect_enabled':False,'clean_truth_used':False,'promotion_authority':False,'software':environment}
identity['implementation_id']=hashlib.sha256(json.dumps({k:identity[k] for k in ['policy_id','policy_manifest_sha256','notebook50_predictions_sha256','predictor_schema_sha256','reconstruction_method','inpaint_radius','crop_margin']},sort_keys=True).encode()).hexdigest()
atomic_json(OUT/'experiment_identity.json',identity); atomic_json(OUT/'software_environment.json',environment)
summary=['NOTEBOOK 53 / V8.8B OPERATIONAL M5_REMOVE BUILDER','',f'Verdict: {verdict}',f'Mode: {INPUT_MODE}',f'Policy ID: {EXPECTED_POLICY_ID}',f'Implementation ID: {identity["implementation_id"]}',f'Pages: {len(page_df)}',f'M3 pixels: {int(page_df.M3_pixels.sum())}',f'SAFE_REMOVE pixels: {int(page_df.SAFE_REMOVE_pixels.sum())}',f'M5_REMOVE pixels: {int(page_df.M5_REMOVE_pixels.sum())}',f'Outside-M3 drift: {int(page_df.outside_M3_drift.sum())}',f'Notebook 50 replay exact: {gates.get("notebook50_replay_exact")}','','No retraining, recalibration, threshold change or SAFE_PROTECT intervention occurred.','PASS authorises Notebook 54 exact-equivalence testing only.']
(OUT/'executive_summary.txt').write_text(chr(10).join(summary))
upload_files=[OUT/'executive_summary.txt',OUT/'notebook53_verdict.json',OUT/'experiment_identity.json',OUT/'software_environment.json',TABLES/'notebook53_gate_results.csv',TABLES/'operational_page_inventory.csv',TABLES/'notebook50_safe_remove_replay.csv',AUDIT/'decision_encoding_audit.csv',INPUTS/'resolved_page_inputs.csv']
rows=[]
for src in upload_files:
    require(src.exists(),f'Missing required upload artefact: {src}')
    dst=UPLOAD/src.name; shutil.copy2(src,dst); rows.append({'file':dst.name,'bytes':dst.stat().st_size,'sha256':sha256_file(dst),'source':str(src.relative_to(OUT))})
pd.DataFrame(rows).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'README.txt').write_text('Notebook 53 V8.8B curated review package. Heavy page images, NPZ masks and parquet decisions remain in the durable run. PASS authorises Notebook 54 equivalence testing only.' + chr(10))
zip_path=UPLOAD/'notebook53_v8_8b_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(UPLOAD.iterdir()):
        if p.is_file() and p!=zip_path:z.write(p,p.name)
require(zip_path.exists() and zip_path.stat().st_size>0,'Upload bundle creation failed.')
print(chr(10).join(summary)); print('Upload bundle:',zip_path)


## Outputs to inspect first

1. `executive_summary.txt`
2. `notebook53_verdict.json`
3. `tables/notebook50_safe_remove_replay.csv`
4. `tables/operational_page_inventory.csv`
5. `audit/decision_encoding_audit.csv`
6. `pages/<page>/corrected.png`
7. `pages/<page>/safe_remove.png`
8. `pages/<page>/m5_remove.png`
9. `upload/notebook53_v8_8b_upload_bundle.zip`

A registered-replay PASS is the expected first result. It proves the frozen SAFE_REMOVE map can be reproduced in the operational wrapper. Notebook 54 must then compare M3, M5_REMOVE and corrected-page arrays against the V8.7 golden masters before any Notebook 09 shadow use.
